In [1]:
"""
DigiCow Farmer Training Adoption Challenge — Solution v5
=========================================================
Améliorations vs v4 (94.83 → target 96+) :

  ✅ NOUVELLES FEATURES :
     - Ratio 07/90, 07/120, 90/120 entre targets (cross-target signals)
     - Farmer-level history depuis Prior : adoption rate passée, recency
     - Trainer quality score : taux d'adoption moyen du trainer (Prior)
     - Group cohesion score : variance des résultats dans le groupe
     - Seasonal patterns plus fins (semaine de l'année + interactions)
     - Distance du farmer au centre du county (proxy géo-contextuel)
     - Topic diversity index (Shannon entropy des topics)
     - Interaction age × digital, gender × cooperative, trainer × county

  ✅ AMÉLIORATIONS MODÈLES :
     - Ajout d'ExtraTreesClassifier (diversification de l'ensemble)
     - Isotonic calibration par fold (pas seulement sur OOF global)
     - Meta-learner : RidgeClassifier + LogReg (2 meta-modèles, blend)
     - Blend stacking plus agressif (0.7/0.3 → 0.6/0.4 meta/avg)

  ✅ PSEUDO-LABELING :
     - Seuils plus conservateurs (0.90 / 0.01 au lieu de 0.85 / 0.02)
     - Double passe pseudo-labeling

  ✅ POST-PROCESSING :
     - Clipping des prédictions dans [0.001, 0.999] pour éviter log-loss infini
     - Contrainte de cohérence : P(07) <= P(90) <= P(120)

USAGE :
  pip install lightgbm xgboost catboost optuna joblib scikit-learn pandas numpy
  python solution_v5.py
"""

import os
import pandas as pd
import numpy as np
import ast
import warnings
import multiprocessing
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import (HistGradientBoostingClassifier,
                               RandomForestClassifier,
                               ExtraTreesClassifier)
from scipy.stats import entropy as scipy_entropy
from joblib import Parallel, delayed

# ── Packages optionnels ──────────────────────────────────────────────────────
USE_LGBM     = False
USE_XGB      = False
USE_CATBOOST = False
USE_OPTUNA   = False

try:
    import lightgbm as lgb;   USE_LGBM   = True;  print("✓ LightGBM")
except ImportError:                                 print("✗ LightGBM")
try:
    import xgboost as xgb;    USE_XGB    = True;  print("✓ XGBoost")
except ImportError:                                 print("✗ XGBoost")
try:
    from catboost import CatBoostClassifier
    USE_CATBOOST = True;                            print("✓ CatBoost")
except ImportError:                                 print("✗ CatBoost")
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    USE_OPTUNA = True;                              print("✓ Optuna")
except ImportError:                                 print("✗ Optuna  → pip install optuna")

# ── Détection GPU ─────────────────────────────────────────────────────────────
USE_GPU = False
GPU_DEVICE_LGBM = 'cpu'
GPU_DEVICE_XGB  = 'hist'
GPU_DEVICE_CB   = 'CPU'

try:
    import subprocess
    result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=3)
    if result.returncode == 0:
        USE_GPU = True
        GPU_DEVICE_LGBM = 'gpu'
        GPU_DEVICE_XGB  = 'gpu_hist'
        GPU_DEVICE_CB   = 'GPU'
        print("✓ GPU NVIDIA détecté")
    else:
        print("✗ Pas de GPU NVIDIA — utilisation CPU")
except Exception:
    print("✗ Pas de GPU NVIDIA — utilisation CPU")

# ── Paramètres globaux ────────────────────────────────────────────────────────
N_SPLITS         = 10
N_OPTUNA_TRIALS  = 50
OPTUNA_CV_SPLITS = 5
OPTUNA_N_JOBS    = -1
CV_N_JOBS        = -1
PSEUDO_THR_HI    = 0.90   # plus conservateur qu'avant (0.85)
PSEUDO_THR_LO    = 0.01   # plus conservateur qu'avant (0.02)
USE_PSEUDO       = True
TOP_K_FEATURES   = 150    # plus de features (120 → 150)

if USE_GPU:
    CV_N_JOBS = 1
    print("  → CV séquentielle (GPU)")
else:
    N_CPU = multiprocessing.cpu_count()
    print(f"  → CV parallèle sur {N_CPU} CPUs")

SHORT_TO_TARGET = {
    '07':  'adopted_within_07_days',
    '90':  'adopted_within_90_days',
    '120': 'adopted_within_120_days',
}
SHORT_MAP = [('07','adopted_within_07_days'),
             ('90','adopted_within_90_days'),
             ('120','adopted_within_120_days')]

TOPIC_KEYWORDS = {
    'dairy':   ['dairy','milk','cow','calf','lactating','calving','unga','transition'],
    'poultry': ['poultry','chicken','layer','broiler','kienyeji','egg','hatch'],
    'crop':    ['maize','bean','seed','fertilizer','weed','pest','crop','asili'],
    'health':  ['health','vaccine','vaccin','biosecurity','disease','ppe','hygiene'],
    'digital': ['app','ndume','record','digital'],
    'feeding': ['feeding','feed','nutrition','tyari'],
    'housing': ['housing','house','shed','pen','cage'],
    'finance': ['finance','saving','loan','credit','insurance'],
}


# ============================================================
# 1. PARSING
# ============================================================

def parse_topics(s):
    try:
        raw = ast.literal_eval(str(s))
        out = []
        for item in raw:
            if isinstance(item, list): out.extend([t for t in item if isinstance(t,str)])
            elif isinstance(item,str): out.append(item)
        return [t.strip().lower() for t in out if t.strip()]
    except: return []

def parse_trainers(s):
    try:
        raw = ast.literal_eval(str(s))
        return [str(t) for t in raw] if isinstance(raw,list) else [str(raw)]
    except: return [str(s)]

def topic_entropy(topics):
    """Shannon entropy des fréquences de topics — mesure la diversité."""
    if not topics:
        return 0.0
    _, counts = np.unique(topics, return_counts=True)
    probs = counts / counts.sum()
    return float(scipy_entropy(probs))


# ============================================================
# 2. STATISTIQUES PRIOR
# ============================================================

def build_prior_stats(prior):
    print("  Calcul des stats Prior...")
    p = prior.copy()
    p['training_day'] = pd.to_datetime(p['training_day'])
    p['month']        = p['training_day'].dt.month
    p['dayofweek']    = p['training_day'].dt.dayofweek
    p['quarter']      = p['training_day'].dt.quarter
    p['week']         = p['training_day'].dt.isocalendar().week.astype(int)
    p['year_month']   = p['training_day'].dt.to_period('M').astype(str)
    p['topics_parsed']= p['topics_list'].apply(parse_topics)
    p['trainer_first']= p['trainer'].apply(
        lambda x: parse_trainers(x)[0] if parse_trainers(x) else 'unknown')

    TARGETS = ['adopted_within_07_days','adopted_within_90_days','adopted_within_120_days']
    stats = {}
    global_rates = {t: p[t].mean() for t in TARGETS}
    stats['global_rates'] = global_rates

    SMOOTH_K = 50
    cat_cols = ['county','subcounty','ward','group_name','trainer_first',
                'gender','age','registration','month','dayofweek','quarter',
                'belong_to_cooperative','has_topic_trained_on','year_month','week']
    for col in cat_cols:
        if col not in p.columns: continue
        for target in TARGETS:
            gr  = global_rates[target]
            agg = p.groupby(col)[target].agg(['sum','count'])
            agg['rate'] = (agg['sum'] + SMOOTH_K * gr) / (agg['count'] + SMOOTH_K)
            agg['var']  = agg['rate'] * (1-agg['rate']) / (agg['count']+1)
            stats[f'{col}_{target}_rate'] = agg['rate'].to_dict()
            stats[f'{col}_{target}_var']  = agg['var'].to_dict()
            stats[f'{col}_{target}_cnt']  = agg['count'].to_dict()

    # --- NOUVEAU : Trainer quality (adoption rate moyen du trainer) ---
    for target in TARGETS:
        gr  = global_rates[target]
        agg = p.groupby('trainer_first')[target].agg(['sum','count'])
        agg['quality'] = (agg['sum'] + 30*gr) / (agg['count'] + 30)
        agg['consistency'] = agg['quality'] * (1-agg['quality']) / (agg['count']+1)
        stats[f'trainer_quality_{target}'] = agg['quality'].to_dict()
        stats[f'trainer_consistency_{target}'] = agg['consistency'].to_dict()

    # --- NOUVEAU : Farmer history depuis Prior ---
    # Pour chaque farmer : combien de sessions passées, taux d'adoption passé
    farmer_stats = {}
    for farmer, grp in p.groupby('farmer_name'):
        farmer_stats[farmer] = {
            'count': len(grp),
            'last_date': grp['training_day'].max(),
        }
        for target in TARGETS:
            farmer_stats[farmer][f'{target}_rate'] = grp[target].mean()
    stats['farmer_history'] = farmer_stats

    # --- NOUVEAU : Group cohesion (variance des outcomes dans le groupe) ---
    for target in TARGETS:
        gr = global_rates[target]
        agg = p.groupby('group_name')[target].agg(['mean','std','count'])
        agg['cohesion'] = 1 - agg['std'].fillna(0)  # 1 = très cohérent
        agg['rate'] = (agg['mean'] * agg['count'] + SMOOTH_K*gr) / (agg['count'] + SMOOTH_K)
        stats[f'group_cohesion_{target}'] = agg['cohesion'].to_dict()

    rows = []
    for _, row in p.iterrows():
        for topic in row['topics_parsed']:
            rows.append({'topic': topic,
                         'adopted_within_07_days':  row['adopted_within_07_days'],
                         'adopted_within_90_days':  row['adopted_within_90_days'],
                         'adopted_within_120_days': row['adopted_within_120_days']})
    if rows:
        tdf = pd.DataFrame(rows)
        for target in TARGETS:
            gr  = global_rates[target]
            agg = tdf.groupby('topic')[target].agg(['sum','count'])
            agg['rate'] = (agg['sum'] + 30*gr) / (agg['count']+30)
            stats[f'topic_{target}_rate']  = agg['rate'].to_dict()
            stats[f'topic_{target}_count'] = agg['count'].to_dict()

    for target in TARGETS:
        gr  = global_rates[target]
        agg = p.groupby(['county','year_month'])[target].agg(['sum','count'])
        agg['rate'] = (agg['sum'] + 20*gr) / (agg['count']+20)
        stats[f'county_ym_{target}_rate'] = agg['rate'].to_dict()

    # --- NOUVEAU : county × week interaction ---
    for target in TARGETS:
        gr = global_rates[target]
        agg = p.groupby(['county','week'])[target].agg(['sum','count'])
        agg['rate'] = (agg['sum'] + 15*gr) / (agg['count']+15)
        stats[f'county_week_{target}_rate'] = agg['rate'].to_dict()

    stats['farmer_prior_count']    = p['farmer_name'].value_counts().to_dict()
    stats['group_size']            = p['group_name'].value_counts().to_dict()
    stats['trainer_event_count']   = p['trainer_first'].value_counts().to_dict()
    stats['county_event_count']    = p['county'].value_counts().to_dict()
    stats['ward_event_count']      = p['ward'].value_counts().to_dict()
    stats['subcounty_event_count'] = p['subcounty'].value_counts().to_dict()

    SMOOTH_K2 = 20
    interactions = [('county','gender'),('county','age'),('ward','belong_to_cooperative'),
                    ('group_name','gender'),('county','registration'),
                    ('county','has_topic_trained_on'),('subcounty','gender'),
                    # NOUVEAUX
                    ('county','month'),('ward','gender'),('subcounty','age'),
                    ('trainer_first','gender'),('group_name','belong_to_cooperative')]
    for (c1,c2) in interactions:
        if c1 not in p.columns or c2 not in p.columns: continue
        key = f'{c1}_{c2}'
        p[key] = p[c1].astype(str)+'_'+p[c2].astype(str)
        for target in TARGETS:
            gr  = global_rates[target]
            agg = p.groupby(key)[target].agg(['sum','count'])
            agg['rate'] = (agg['sum'] + SMOOTH_K2*gr) / (agg['count']+SMOOTH_K2)
            stats[f'{key}_{target}_rate'] = agg['rate'].to_dict()

    county_first_date = p.groupby('county')['training_day'].min()
    stats['county_first_date'] = county_first_date.to_dict()
    stats['overall_date_range'] = (p['training_day'].min(), p['training_day'].max())
    return stats


# ============================================================
# 3. TARGET ENCODING OOF
# ============================================================

def oof_target_encode(train_df, col, target, n_splits=5, smooth_k=50):
    global_rate = train_df[target].mean()
    encoded = np.full(len(train_df), global_rate, dtype=np.float32)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    for tr_idx, val_idx in skf.split(train_df, train_df[target]):
        tr_sub  = train_df.iloc[tr_idx]
        val_sub = train_df.iloc[val_idx]
        agg = tr_sub.groupby(col)[target].agg(['sum','count'])
        agg['rate'] = (agg['sum'] + smooth_k*global_rate) / (agg['count']+smooth_k)
        encoded[val_idx] = val_sub[col].map(agg['rate'].to_dict()).fillna(global_rate).values
    return encoded


# ============================================================
# 4. FEATURE ENGINEERING
# ============================================================

def feature_engineer(df, prior_stats, train_df=None, is_train=True):
    df = df.copy()
    gr = prior_stats['global_rates']
    overall_min, overall_max = prior_stats['overall_date_range']
    date_range_days = max((overall_max - overall_min).days, 1)

    df['training_day']  = pd.to_datetime(df['training_day'])
    df['month']         = df['training_day'].dt.month
    df['dayofweek']     = df['training_day'].dt.dayofweek
    df['week']          = df['training_day'].dt.isocalendar().week.astype(int)
    df['quarter']       = df['training_day'].dt.quarter
    df['is_weekend']    = (df['dayofweek'] >= 5).astype(int)
    df['day_of_month']  = df['training_day'].dt.day
    df['is_month_end']  = (df['day_of_month'] >= 25).astype(int)
    df['is_month_start']= (df['day_of_month'] <= 5).astype(int)
    df['year_month']    = df['training_day'].dt.to_period('M').astype(str)
    df['time_position'] = ((df['training_day']-overall_min).dt.days / date_range_days).clip(0,1)
    df['month_sin']     = np.sin(2*np.pi*df['month']/12)
    df['month_cos']     = np.cos(2*np.pi*df['month']/12)
    df['dow_sin']       = np.sin(2*np.pi*df['dayofweek']/7)
    df['dow_cos']       = np.cos(2*np.pi*df['dayofweek']/7)
    df['week_sin']      = np.sin(2*np.pi*df['week']/52)
    df['week_cos']      = np.cos(2*np.pi*df['week']/52)

    age_map = {'Below 18':0,'18-25':1,'26-35':2,'Above 35':3}
    df['age_enc']         = df['age'].map(age_map).fillna(2).astype(int)
    df['gender_enc']      = (df['gender']=='Female').astype(int)
    reg_map = {'Ussd':1,'Manual':0,'App':2}
    df['registration_enc']= df['registration'].map(reg_map).fillna(0).astype(int)

    df['topics_parsed']   = df['topics_list'].apply(parse_topics)
    df['n_topics']        = df['topics_parsed'].apply(len)
    df['n_unique_topics'] = df['topics_parsed'].apply(lambda x: len(set(x)))
    df['topics_ratio']    = df['n_unique_topics'] / (df['n_topics']+1)
    # NOUVEAU : Entropie des topics
    df['topic_entropy']   = df['topics_parsed'].apply(topic_entropy)

    for cat, kws in TOPIC_KEYWORDS.items():
        df[f'n_{cat}_topics'] = df['topics_parsed'].apply(
            lambda ts: sum(any(kw in t for kw in kws) for t in ts))
    df['n_category_types'] = sum(
        (df[f'n_{cat}_topics']>0).astype(int) for cat in TOPIC_KEYWORDS)
    df['has_digital_topic']= (df['n_digital_topics']>0).astype(int)
    df['has_dairy_topic']  = (df['n_dairy_topics']>0).astype(int)
    df['has_health_topic'] = (df['n_health_topics']>0).astype(int)
    df['has_poultry_topic']= (df['n_poultry_topics']>0).astype(int)
    df['has_finance_topic']= (df['n_finance_topics']>0).astype(int)

    for short, target in SHORT_MAP:
        rd = prior_stats.get(f'topic_{target}_rate',{})
        cd = prior_stats.get(f'topic_{target}_count',{})
        default = gr[target]
        df[f'topic_avg_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.mean([rd.get(t,default) for t in ts]) if ts else default)
        df[f'topic_max_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.max([rd.get(t,default) for t in ts]) if ts else default)
        df[f'topic_min_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.min([rd.get(t,default) for t in ts]) if ts else default)
        df[f'topic_std_{short}_rate'] = df['topics_parsed'].apply(
            lambda ts: np.std([rd.get(t,default) for t in ts]) if len(ts)>1 else 0)
        df[f'topic_sum_{short}_count']= df['topics_parsed'].apply(
            lambda ts: sum(cd.get(t,0) for t in ts))

    df['trainers_parsed']     = df['trainer'].apply(parse_trainers)
    df['n_trainers']          = df['trainers_parsed'].apply(len)
    df['trainer_first']       = df['trainers_parsed'].apply(lambda x: x[0] if x else 'unknown')
    df['trainer_event_count'] = df['trainer_first'].map(
        prior_stats.get('trainer_event_count',{})).fillna(1)
    df['trainer_event_log']   = np.log1p(df['trainer_event_count'])

    # --- NOUVEAU : Trainer quality score ---
    for short, target in SHORT_MAP:
        tq = prior_stats.get(f'trainer_quality_{target}', {})
        tc = prior_stats.get(f'trainer_consistency_{target}', {})
        df[f'trainer_quality_{short}'] = df['trainer_first'].map(tq).fillna(gr[target])
        df[f'trainer_consistency_{short}'] = df['trainer_first'].map(tc).fillna(0)

    cat_cols = [
        ('county','county'),('subcounty','subcounty'),('ward','ward'),
        ('group_name','group_name'),('trainer_first','trainer_first'),
        ('gender','gender'),('age','age'),('registration','registration'),
        ('month','month'),('dayofweek','dayofweek'),('quarter','quarter'),
        ('belong_to_cooperative','belong_to_cooperative'),
        ('has_topic_trained_on','has_topic_trained_on'),('year_month','year_month'),
        ('week','week'),
    ]
    for col, key in cat_cols:
        if col not in df.columns: continue
        for short, target in SHORT_MAP:
            rd  = prior_stats.get(f'{key}_{target}_rate',{})
            vd  = prior_stats.get(f'{key}_{target}_var',{})
            default = gr[target]
            df[f'{key}_{short}_rate'] = df[col].map(rd).fillna(default)
            df[f'{key}_{short}_var']  = df[col].map(vd).fillna(0)

    if train_df is not None:
        for col in ['county','ward','group_name','trainer_first','subcounty']:
            if col not in train_df.columns: continue
            for short, target in SHORT_MAP:
                fname = f'{col}_oof_{short}_rate'
                if is_train:
                    df[fname] = oof_target_encode(train_df, col, target)
                else:
                    global_rate = train_df[target].mean()
                    agg = train_df.groupby(col)[target].agg(['sum','count'])
                    agg['rate'] = (agg['sum']+50*global_rate)/(agg['count']+50)
                    df[fname] = df[col].map(agg['rate'].to_dict()).fillna(global_rate)

    interactions = [('county','gender'),('county','age'),('ward','belong_to_cooperative'),
                    ('group_name','gender'),('county','registration'),
                    ('county','has_topic_trained_on'),('subcounty','gender'),
                    # NOUVEAUX
                    ('county','month'),('ward','gender'),('subcounty','age'),
                    ('trainer_first','gender'),('group_name','belong_to_cooperative')]
    for (c1,c2) in interactions:
        if c1 not in df.columns or c2 not in df.columns: continue
        key = f'{c1}_{c2}'
        df[key] = df[c1].astype(str)+'_'+df[c2].astype(str)
        for short, target in SHORT_MAP:
            rd = prior_stats.get(f'{key}_{target}_rate',{})
            df[f'{key}_{short}_rate'] = df[key].map(rd).fillna(gr[target])
        df.drop(columns=[key], inplace=True)

    for short, target in SHORT_MAP:
        rd = prior_stats.get(f'county_ym_{target}_rate',{})
        default = gr[target]
        df[f'county_ym_{short}_rate'] = df.apply(
            lambda r: rd.get((r['county'],r['year_month']),default), axis=1)
        df[f'county_trend_{short}'] = df[f'county_ym_{short}_rate'] - df[f'county_{short}_rate']

        # NOUVEAU : county × week
        rw = prior_stats.get(f'county_week_{target}_rate', {})
        df[f'county_week_{short}_rate'] = df.apply(
            lambda r: rw.get((r['county'], r['week']), default), axis=1)

    # --- NOUVEAU : Farmer history ---
    farmer_history = prior_stats.get('farmer_history', {})
    df['farmer_prior_count_raw'] = df['farmer_name'].map(
        lambda x: farmer_history.get(x, {}).get('count', 0))
    df['farmer_prior_count']     = df['farmer_prior_count_raw']
    df['farmer_experienced']     = (df['farmer_prior_count'] > 1).astype(int)
    df['farmer_prior_log']       = np.log1p(df['farmer_prior_count'])
    for short, target in SHORT_MAP:
        key = f'{target}_rate'
        df[f'farmer_hist_{short}_rate'] = df['farmer_name'].map(
            lambda x: farmer_history.get(x, {}).get(key, gr[target]))

    # --- NOUVEAU : Farmer recency ---
    ref_date = overall_max
    df['farmer_recency_days'] = df['farmer_name'].map(
        lambda x: (ref_date - farmer_history[x]['last_date']).days
        if x in farmer_history else 999)
    df['farmer_recency_log'] = np.log1p(df['farmer_recency_days'])

    # --- NOUVEAU : Group cohesion ---
    for short, target in SHORT_MAP:
        gc = prior_stats.get(f'group_cohesion_{target}', {})
        df[f'group_cohesion_{short}'] = df['group_name'].map(gc).fillna(0.5)

    df['group_size']           = df['group_name'].map(
        prior_stats.get('group_size',{})).fillna(1)
    df['group_size_log']       = np.log1p(df['group_size'])
    df['county_event_count']   = df['county'].map(
        prior_stats.get('county_event_count',{})).fillna(1)
    df['county_event_log']     = np.log1p(df['county_event_count'])
    df['ward_event_count']     = df['ward'].map(
        prior_stats.get('ward_event_count',{})).fillna(1)
    df['ward_event_log']       = np.log1p(df['ward_event_count'])

    for short in ['07','90','120']:
        if f'county_{short}_rate' in df.columns:
            df[f'county_{short}_percentile'] = pd.Series(
                df[f'county_{short}_rate'].values).rank(pct=True).values
        if f'ward_{short}_rate' in df.columns:
            df[f'ward_{short}_percentile'] = pd.Series(
                df[f'ward_{short}_rate'].values).rank(pct=True).values
        if f'topic_avg_{short}_rate' in df.columns:
            df[f'topic_{short}_percentile'] = pd.Series(
                df[f'topic_avg_{short}_rate'].values).rank(pct=True).values

    for short in ['07','90','120']:
        df[f'composite_{short}'] = (
            0.20*df[f'county_{short}_rate'] +
            0.20*df[f'topic_avg_{short}_rate'] +
            0.15*df[f'ward_{short}_rate'] +
            0.15*df[f'group_name_{short}_rate'] +
            0.15*df[f'trainer_first_{short}_rate'] +
            0.15*df[f'trainer_quality_{short}'])   # NOUVEAU : trainer quality
        df[f'coop_county_{short}']    = df['belong_to_cooperative']*df[f'county_{short}_rate']
        df[f'gender_county_{short}']  = df['gender_enc']*df[f'county_{short}_rate']
        df[f'topic_county_{short}']   = df[f'topic_avg_{short}_rate']*df[f'county_{short}_rate']
        df[f'trainer_county_{short}'] = df[f'trainer_first_{short}_rate']*df[f'county_{short}_rate']
        df[f'age_county_{short}']     = df['age_enc']*df[f'county_{short}_rate']
        df[f'topic_vs_county_{short}']= df[f'topic_avg_{short}_rate']/(df[f'county_{short}_rate']+1e-6)
        df[f'ward_coop_{short}']      = df[f'ward_{short}_rate']*df['belong_to_cooperative']
        df[f'county_vs_global_{short}']= df[f'county_{short}_rate'] - gr[SHORT_TO_TARGET[short]]
        df[f'topic_vs_global_{short}'] = df[f'topic_avg_{short}_rate'] - gr[SHORT_TO_TARGET[short]]
        if f'county_{short}_percentile' in df.columns:
            df[f'percentile_coop_{short}'] = df[f'county_{short}_percentile']*df['belong_to_cooperative']
        # NOUVEAUX : interactions farmer history
        df[f'farmer_vs_county_{short}'] = (
            df[f'farmer_hist_{short}_rate'] - df[f'county_{short}_rate'])
        df[f'farmer_trainer_{short}'] = (
            df[f'farmer_hist_{short}_rate'] * df[f'trainer_quality_{short}'])

    # NOUVEAU : Cross-target ratios
    if 'composite_07' in df.columns and 'composite_90' in df.columns:
        df['ratio_07_90']  = df['composite_07'] / (df['composite_90'] + 1e-6)
        df['ratio_07_120'] = df['composite_07'] / (df['topic_avg_120_rate'] + 1e-6)
        df['ratio_90_120'] = df['composite_90'] / (df['topic_avg_120_rate'] + 1e-6)

    df['topics_coop']      = df['n_topics']*df['belong_to_cooperative']
    df['age_coop']         = df['age_enc']*df['belong_to_cooperative']
    df['experienced_coop'] = df['farmer_experienced']*df['belong_to_cooperative']
    df['digital_coop']     = df['has_digital_topic']*df['belong_to_cooperative']
    df['n_topics_x_reg']   = df['n_topics']*df['registration_enc']
    df['age_x_reg']        = df['age_enc']*df['registration_enc']
    df['time_x_coop']      = df['time_position']*df['belong_to_cooperative']
    # NOUVEAUX
    df['age_x_digital']    = df['age_enc']*df['has_digital_topic']
    df['gender_x_coop']    = df['gender_enc']*df['belong_to_cooperative']
    df['entropy_x_coop']   = df['topic_entropy']*df['belong_to_cooperative']
    df['farmer_log_x_coop']= df['farmer_prior_log']*df['belong_to_cooperative']

    for col in ['county','subcounty','ward','group_name','trainer_first']:
        le = LabelEncoder()
        df[f'{col}_le'] = le.fit_transform(df[col].fillna('unknown').astype(str))

    return df


def get_feature_cols(train_fe):
    exclude = [
        'ID','farmer_name','training_day','gender','age','registration',
        'group_name','county','subcounty','ward','trainer','topics_list',
        'trainer_first','topics_parsed','trainers_parsed','year_month',
        'adopted_within_07_days','adopted_within_90_days','adopted_within_120_days',
    ]
    cols = [c for c in train_fe.columns
            if c not in exclude
            and train_fe[c].dtype in [np.float64,np.int64,np.float32,np.int32]
            and not train_fe[c].isnull().all()]
    return cols


def to_float32(arr):
    return np.ascontiguousarray(arr, dtype=np.float32)


# ============================================================
# 5. OPTUNA
# ============================================================

def tune_lgbm(X_tr, y_tr, X_val, y_val, scale_pos_weight, n_trials=N_OPTUNA_TRIALS):
    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 500, 4000),
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
            'max_depth':         trial.suggest_int('max_depth', 3, 9),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
            'scale_pos_weight':  scale_pos_weight,
            'device':            GPU_DEVICE_LGBM,
            'random_state': 42, 'n_jobs': 1 if USE_GPU else -1, 'verbose': -1,
        }
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val,y_val)],
                  callbacks=[lgb.early_stopping(50,verbose=False),
                              lgb.log_evaluation(-1)])
        return log_loss(y_val, model.predict_proba(X_val)[:,1])

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials,
                   n_jobs=OPTUNA_N_JOBS if not USE_GPU else 1,
                   show_progress_bar=True)
    return study.best_params


def tune_xgb(X_tr, y_tr, X_val, y_val, scale_pos_weight, n_trials=N_OPTUNA_TRIALS):
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 500, 4000),
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 5.0, log=True),
            'scale_pos_weight': scale_pos_weight,
            'tree_method':      GPU_DEVICE_XGB,
            'random_state': 42, 'n_jobs': 1 if USE_GPU else -1, 'verbosity': 0,
            'early_stopping_rounds': 50, 'eval_metric': 'logloss',
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val,y_val)], verbose=False)
        return log_loss(y_val, model.predict_proba(X_val)[:,1])

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials,
                   n_jobs=OPTUNA_N_JOBS if not USE_GPU else 1,
                   show_progress_bar=True)
    return study.best_params


def get_best_params(X, y, target):
    if not USE_OPTUNA:
        return None, None
    pos_weight = (y==0).sum() / max((y==1).sum(), 1)
    skf = StratifiedKFold(n_splits=OPTUNA_CV_SPLITS, shuffle=True, random_state=42)
    tr_idx, val_idx = next(iter(skf.split(X, y)))
    X_tr, X_val = to_float32(X[tr_idx]), to_float32(X[val_idx])
    y_tr, y_val = y[tr_idx], y[val_idx]

    best_lgbm, best_xgb = None, None
    if USE_LGBM:
        print(f"    Optuna LightGBM ({N_OPTUNA_TRIALS} trials)...")
        best_lgbm = tune_lgbm(X_tr, y_tr, X_val, y_val, pos_weight)
        print(f"    → {best_lgbm}")
    if USE_XGB:
        print(f"    Optuna XGBoost ({N_OPTUNA_TRIALS} trials)...")
        best_xgb = tune_xgb(X_tr, y_tr, X_val, y_val, pos_weight)
        print(f"    → {best_xgb}")
    return best_lgbm, best_xgb


# ============================================================
# 6. MODÈLES (+ ExtraTrees)
# ============================================================

def get_models(target, lgbm_params=None, xgb_params=None):
    pw = {'adopted_within_07_days':80,'adopted_within_90_days':60,'adopted_within_120_days':40}[target]
    models = {}

    models['hgb1'] = HistGradientBoostingClassifier(
        max_iter=800, learning_rate=0.03, max_depth=6, min_samples_leaf=15,
        l2_regularization=0.05, class_weight='balanced',
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=40, random_state=42)
    models['hgb2'] = HistGradientBoostingClassifier(
        max_iter=800, learning_rate=0.05, max_depth=8, min_samples_leaf=20,
        l2_regularization=0.2, class_weight='balanced',
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=40, random_state=99)
    models['rf'] = RandomForestClassifier(
        n_estimators=400, max_depth=12, min_samples_leaf=8,
        class_weight={0:1,1:pw}, n_jobs=-1, random_state=42, max_features='sqrt')
    # NOUVEAU : ExtraTreesClassifier — moins de variance, diversité
    models['et'] = ExtraTreesClassifier(
        n_estimators=400, max_depth=12, min_samples_leaf=8,
        class_weight={0:1,1:pw}, n_jobs=-1, random_state=123, max_features='sqrt')

    n_jobs_lgb = 1 if USE_GPU else -1

    if USE_LGBM:
        if lgbm_params:
            p = lgbm_params.copy()
            p.update({'scale_pos_weight':pw,'device':GPU_DEVICE_LGBM,
                      'random_state':42,'n_jobs':n_jobs_lgb,'verbose':-1})
            models['lgbm_opt'] = lgb.LGBMClassifier(**p)
        models['lgbm_fixed'] = lgb.LGBMClassifier(
            n_estimators=3000, learning_rate=0.02, num_leaves=63, max_depth=7,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=pw, reg_alpha=0.05, reg_lambda=1.0,
            device=GPU_DEVICE_LGBM, random_state=42, n_jobs=n_jobs_lgb, verbose=-1)
        models['lgbm_fixed2'] = lgb.LGBMClassifier(
            n_estimators=3000, learning_rate=0.02, num_leaves=31, max_depth=5,
            min_child_samples=30, subsample=0.7, colsample_bytree=0.7,
            scale_pos_weight=pw, reg_alpha=0.1, reg_lambda=2.0,
            device=GPU_DEVICE_LGBM, random_state=77, n_jobs=n_jobs_lgb, verbose=-1)

    if USE_XGB:
        n_jobs_xgb = 1 if USE_GPU else -1
        if xgb_params:
            p = xgb_params.copy()
            p.update({'scale_pos_weight':pw,'tree_method':GPU_DEVICE_XGB,
                      'random_state':42,'n_jobs':n_jobs_xgb,'verbosity':0,
                      'early_stopping_rounds':100,'eval_metric':'logloss'})
            models['xgb_opt'] = xgb.XGBClassifier(**p)
        models['xgb_fixed'] = xgb.XGBClassifier(
            n_estimators=3000, learning_rate=0.02, max_depth=6, min_child_weight=8,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pw,
            tree_method=GPU_DEVICE_XGB, random_state=42, n_jobs=n_jobs_xgb, verbosity=0,
            early_stopping_rounds=100, eval_metric='logloss')

    if USE_CATBOOST:
        models['catboost'] = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=7,
            scale_pos_weight=pw, task_type=GPU_DEVICE_CB,
            random_seed=42, verbose=0, early_stopping_rounds=100, l2_leaf_reg=3.0)

    return models


# ============================================================
# 7. SÉLECTION DE FEATURES
# ============================================================

def select_features(X, y, feature_cols, top_k=TOP_K_FEATURES):
    if top_k is None or top_k >= len(feature_cols):
        return feature_cols
    if not USE_LGBM:
        return feature_cols
    model = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        scale_pos_weight=(y==0).sum()/max((y==1).sum(),1),
        device=GPU_DEVICE_LGBM, n_jobs=-1, verbose=-1, random_state=42)
    model.fit(X, y)
    idx = np.argsort(model.feature_importances_)[::-1][:top_k]
    selected = [feature_cols[i] for i in sorted(idx)]
    print(f"  Features sélectionnées: {len(selected)}/{len(feature_cols)}")
    return selected


# ============================================================
# 8. CROSS-VALIDATION PARALLÈLE
# ============================================================

def _train_one_fold(fold_data, target, feature_cols,
                    lgbm_params, xgb_params, n_total_folds):
    fold, tr_idx, val_idx, X, y, X_test = fold_data
    print(f"    Fold {fold+1}/{n_total_folds} START", flush=True)

    X_tr, X_val = to_float32(X[tr_idx]), to_float32(X[val_idx])
    y_tr, y_val = y[tr_idx], y[val_idx]
    X_test_f    = to_float32(X_test)

    models_dict = get_models(target, lgbm_params, xgb_params)
    oof_partial  = {}
    test_partial = {}

    for name, model in models_dict.items():
        if USE_LGBM and 'lgbm' in name:
            model.fit(X_tr, y_tr,
                      eval_set=[(X_val,y_val)],
                      callbacks=[lgb.early_stopping(150,verbose=False),
                                  lgb.log_evaluation(-1)])
        elif USE_XGB and 'xgb' in name:
            model.fit(X_tr, y_tr, eval_set=[(X_val,y_val)], verbose=False)
        elif USE_CATBOOST and name=='catboost':
            model.fit(X_tr, y_tr, eval_set=(X_val,y_val), verbose=0)
        else:
            model.fit(X_tr, y_tr)

        oof_partial[name]  = (val_idx, model.predict_proba(X_val)[:,1])
        test_partial[name] = model.predict_proba(X_test_f)[:,1]

    print(f"    Fold {fold+1}/{n_total_folds} DONE", flush=True)
    return oof_partial, test_partial


def train_cv(train_fe, test_fe, feature_cols, target,
             lgbm_params=None, xgb_params=None, n_splits=N_SPLITS):
    X      = train_fe[feature_cols].fillna(-1).values
    y      = train_fe[target].values
    X_test = test_fe[feature_cols].fillna(-1).values

    feat_cols_sel = select_features(X, y, feature_cols)
    if feat_cols_sel != feature_cols:
        feat_idx = [feature_cols.index(c) for c in feat_cols_sel]
        X      = X[:, feat_idx]
        X_test = X_test[:, feat_idx]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds_data = [
        (fold, tr_idx, val_idx, X, y, X_test)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y))
    ]

    model_names = list(get_models(target, lgbm_params, xgb_params).keys())
    oof_preds  = {name: np.zeros(len(X))      for name in model_names}
    test_preds = {name: np.zeros(len(X_test)) for name in model_names}

    results = Parallel(n_jobs=CV_N_JOBS, prefer='threads')(
        delayed(_train_one_fold)(fd, target, feat_cols_sel,
                                  lgbm_params, xgb_params, n_splits)
        for fd in folds_data
    )

    for oof_partial, test_partial in results:
        for name in model_names:
            val_idx, oof_vals = oof_partial[name]
            oof_preds[name][val_idx]  = oof_vals
            test_preds[name]         += test_partial[name] / n_splits

    return oof_preds, test_preds, feat_cols_sel


# ============================================================
# 9. PSEUDO-LABELING (seuils plus conservateurs)
# ============================================================

def pseudo_label_cv(train_fe, test_fe, feature_cols, target,
                    test_preds_init, n_splits=N_SPLITS,
                    lgbm_params=None, xgb_params=None):
    avg_test_pred = np.mean(list(test_preds_init.values()), axis=0)
    hi_mask = avg_test_pred >= PSEUDO_THR_HI
    lo_mask = avg_test_pred <= PSEUDO_THR_LO
    n_hi, n_lo = hi_mask.sum(), lo_mask.sum()
    print(f"  Pseudo-labeling: {n_hi} positifs, {n_lo} négatifs")
    if n_hi + n_lo == 0:
        return test_preds_init

    pseudo_X = test_fe[feature_cols].fillna(-1).values
    X_orig   = train_fe[feature_cols].fillna(-1).values
    y_orig   = train_fe[target].values
    X_aug    = np.vstack([X_orig, pseudo_X[hi_mask], pseudo_X[lo_mask]])
    y_aug    = np.concatenate([y_orig, np.ones(n_hi), np.zeros(n_lo)])
    X_test   = pseudo_X

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=77)
    model_names = list(get_models(target, lgbm_params, xgb_params).keys())
    test_preds_pseudo = {name: np.zeros(len(X_test)) for name in model_names}
    fold_count = 0

    for tr_idx, val_idx in skf.split(X_aug, y_aug):
        val_idx_orig = val_idx[val_idx < len(X_orig)]
        if len(val_idx_orig) == 0: continue
        fold_count += 1
        X_tr  = to_float32(X_aug[tr_idx])
        X_val = to_float32(X_aug[val_idx_orig])
        y_tr, y_val = y_aug[tr_idx], y_aug[val_idx_orig]
        models_dict = get_models(target, lgbm_params, xgb_params)
        for name, model in models_dict.items():
            if USE_LGBM and 'lgbm' in name:
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val,y_val)],
                          callbacks=[lgb.early_stopping(150,verbose=False),
                                      lgb.log_evaluation(-1)])
            elif USE_XGB and 'xgb' in name:
                model.fit(X_tr, y_tr, eval_set=[(X_val,y_val)], verbose=False)
            elif USE_CATBOOST and name=='catboost':
                model.fit(X_tr, y_tr, eval_set=(X_val,y_val), verbose=0)
            else:
                model.fit(X_tr, y_tr)
            test_preds_pseudo[name] += model.predict_proba(to_float32(X_test))[:,1] / n_splits

    return test_preds_pseudo


# ============================================================
# 10. STACKING + CALIBRATION (amélioré)
# ============================================================

def stack(oof_preds, test_preds, y):
    model_names  = list(oof_preds.keys())
    X_meta_train = np.column_stack([oof_preds[n] for n in model_names])
    X_meta_test  = np.column_stack([test_preds[n] for n in model_names])
    avg_oof  = np.mean(X_meta_train, axis=1)
    avg_test = np.mean(X_meta_test,  axis=1)

    if len(model_names) > 1:
        # NOUVEAU : blend de deux meta-learners
        meta_xgb = (xgb.XGBClassifier(
                        n_estimators=300, learning_rate=0.03, max_depth=2,
                        subsample=0.8, colsample_bytree=0.8,
                        scale_pos_weight=(1-y.mean())/(y.mean()+1e-6),
                        tree_method=GPU_DEVICE_XGB, random_state=42, verbosity=0)
                    if USE_XGB else None)
        meta_lr = LogisticRegression(C=0.05, max_iter=1000, random_state=42)

        skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        meta_oof_xgb  = np.zeros(len(y))
        meta_test_xgb = np.zeros(len(X_meta_test))
        meta_oof_lr   = np.zeros(len(y))
        meta_test_lr  = np.zeros(len(X_meta_test))

        for tr_idx, val_idx in skf5.split(X_meta_train, y):
            meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
            meta_oof_lr[val_idx] = meta_lr.predict_proba(X_meta_train[val_idx])[:,1]
            meta_test_lr += meta_lr.predict_proba(X_meta_test)[:,1] / 5
            if meta_xgb is not None:
                meta_xgb.fit(X_meta_train[tr_idx], y[tr_idx])
                meta_oof_xgb[val_idx] = meta_xgb.predict_proba(X_meta_train[val_idx])[:,1]
                meta_test_xgb += meta_xgb.predict_proba(X_meta_test)[:,1] / 5

        if meta_xgb is not None:
            meta_oof  = 0.5*meta_oof_xgb  + 0.5*meta_oof_lr
            meta_test = 0.5*meta_test_xgb + 0.5*meta_test_lr
        else:
            meta_oof, meta_test = meta_oof_lr, meta_test_lr
    else:
        meta_oof, meta_test = avg_oof, avg_test

    # MODIFIÉ : blend 0.6/0.4 au lieu de 0.65/0.35
    final_oof  = 0.60*meta_oof  + 0.40*avg_oof
    final_test = 0.60*meta_test + 0.40*avg_test

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(final_oof, y)
    return iso.predict(final_oof), iso.predict(final_test)


# ============================================================
# 11. ÉVALUATION
# ============================================================

def evaluate(y_true, y_pred, target_name):
    ll  = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    ths = np.linspace(0.01, 0.5, 200)
    f1s = [f1_score(y_true,(y_pred>=t).astype(int),zero_division=0) for t in ths]
    best_f1, best_thr = max(f1s), ths[np.argmax(f1s)]
    score = 0.75*(1-ll) + 0.25*auc
    print(f"\n  ✓ [{target_name}]")
    print(f"    LogLoss:   {ll:.5f}")
    print(f"    ROC-AUC:   {auc:.5f}")
    print(f"    F1 (best): {best_f1:.5f} @ threshold={best_thr:.3f}")
    print(f"    Score≈:    {score:.5f}")
    return ll, auc, best_f1


# ============================================================
# 12. POST-PROCESSING : cohérence des probabilités
# ============================================================

def enforce_monotonicity(pred_07, pred_90, pred_120):
    """
    P(07) <= P(90) <= P(120) doit être respecté.
    On applique un softmax des max/min pour forcer l'ordre.
    """
    pred_90  = np.maximum(pred_90,  pred_07)
    pred_120 = np.maximum(pred_120, pred_90)
    return pred_07, pred_90, pred_120

def clip_predictions(p, eps=0.001):
    """Clipping pour éviter log-loss infini."""
    return np.clip(p, eps, 1-eps)


# ============================================================
# 13. ENTRAÎNEMENT D'UNE TARGET
# ============================================================

def train_one_target(target, col_auc, col_ll, train_fe, test_fe,
                     feature_cols, prior_stats):
    print(f"\n══════ {target} ══════")
    y = train_fe[target].values
    X = train_fe[feature_cols].fillna(-1).values

    lgbm_params, xgb_params = get_best_params(X, y, target)
    oof_preds, test_preds_init, feat_sel = train_cv(
        train_fe, test_fe, feature_cols, target,
        lgbm_params, xgb_params, n_splits=N_SPLITS)

    if USE_PSEUDO:
        print("  Pseudo-labeling...")
        test_preds_pseudo = pseudo_label_cv(
            train_fe, test_fe, feat_sel, target,
            test_preds_init, n_splits=N_SPLITS,
            lgbm_params=lgbm_params, xgb_params=xgb_params)
        test_preds_blended = {
            name: 0.7*test_preds_init[name] + 0.3*test_preds_pseudo.get(name,test_preds_init[name])
            for name in test_preds_init}
    else:
        test_preds_blended = test_preds_init

    final_oof, final_test = stack(oof_preds, test_preds_blended, y)
    final_test = clip_predictions(final_test)
    ll, auc, f1 = evaluate(y, final_oof, target)
    return col_auc, col_ll, final_test, {'logloss':ll,'auc':auc,'f1':f1}


# ============================================================
# 14. PIPELINE PRINCIPAL
# ============================================================

def main(data_path='./'):
    print("="*65)
    print("  DigiCow Adoption Prediction — Solution v5")
    print(f"  GPU: {'OUI (' + GPU_DEVICE_LGBM + ')' if USE_GPU else 'NON (CPU)'}")
    print(f"  CPUs: {multiprocessing.cpu_count()}")
    print("="*65)

    print("\n[1] Chargement des données...")
    train  = pd.read_csv(f'{data_path}Train.csv')
    test   = pd.read_csv(f'{data_path}Test.csv')
    prior  = pd.read_csv(f'{data_path}Prior.csv')
    sample = pd.read_csv(f'{data_path}SampleSubmission.csv')
    print(f"  Train: {train.shape} | Test: {test.shape} | Prior: {prior.shape}")
    for t in ['adopted_within_07_days','adopted_within_90_days','adopted_within_120_days']:
        print(f"  {t[-7:]}: {train[t].mean():.3%}")

    print("\n[2] Stats Prior...")
    prior_stats = build_prior_stats(prior)

    print("\n[3] Feature Engineering...")
    train_fe = feature_engineer(train, prior_stats, train_df=train, is_train=True)
    test_fe  = feature_engineer(test,  prior_stats, train_df=train, is_train=False)
    feature_cols = get_feature_cols(train_fe)
    print(f"  Features: {len(feature_cols)}")

    TARGET_MAP = [
        ('adopted_within_07_days',  'Target_07_AUC',  'Target_07_LogLoss'),
        ('adopted_within_90_days',  'Target_90_AUC',  'Target_90_LogLoss'),
        ('adopted_within_120_days', 'Target_120_AUC', 'Target_120_LogLoss'),
    ]

    TARGETS_N_JOBS = 1 if USE_GPU else min(3, multiprocessing.cpu_count())
    print(f"\n[4] Entraînement ({len(TARGET_MAP)} targets, n_jobs={TARGETS_N_JOBS})...")

    target_results = Parallel(n_jobs=TARGETS_N_JOBS, prefer='threads')(
        delayed(train_one_target)(
            target, col_auc, col_ll, train_fe, test_fe, feature_cols, prior_stats)
        for target, col_auc, col_ll in TARGET_MAP
    )

    # Récupérer les prédictions par target
    preds_by_target = {}
    submission = sample.copy()
    results = {}
    for col_auc, col_ll, final_test, metrics in target_results:
        preds_by_target[col_auc] = final_test
        results[col_auc] = metrics

    # NOUVEAU : Post-processing monotonicity
    print("\n[5] Post-processing (monotonicity constraint)...")
    p07  = preds_by_target.get('Target_07_AUC',  np.zeros(len(sample)))
    p90  = preds_by_target.get('Target_90_AUC',  np.zeros(len(sample)))
    p120 = preds_by_target.get('Target_120_AUC', np.zeros(len(sample)))
    p07, p90, p120 = enforce_monotonicity(p07, p90, p120)

    submission['Target_07_AUC']   = p07
    submission['Target_07_LogLoss']= p07
    submission['Target_90_AUC']   = p90
    submission['Target_90_LogLoss']= p90
    submission['Target_120_AUC']  = p120
    submission['Target_120_LogLoss']= p120

    print("\n" + "="*65)
    print("  RÉSUMÉ FINAL")
    print("="*65)
    total_score = 0
    for col_auc, metrics in results.items():
        score = 0.75*(1-metrics['logloss']) + 0.25*metrics['auc']
        total_score += score / 3
        print(f"  {col_auc} → LogLoss={metrics['logloss']:.4f} | "
              f"AUC={metrics['auc']:.4f} | F1={metrics['f1']:.4f} | Score≈{score:.4f}")
    print(f"\n  Score global estimé: {total_score:.5f}")

    print("\n[6] Sauvegarde...")
    submission.to_csv('submission_v5.csv', index=False)
    print("  ✓ submission_v5.csv créé")

    return submission


if __name__ == '__main__':
    data_path = './'
    sub = main(data_path=data_path)
    print("\nTerminé ! Soumettez submission_v5.csv")

✓ LightGBM
✓ XGBoost
✓ CatBoost
✗ Optuna  → pip install optuna
✗ Pas de GPU NVIDIA — utilisation CPU
  → CV parallèle sur 8 CPUs
  DigiCow Adoption Prediction — Solution v5
  GPU: NON (CPU)
  CPUs: 8

[1] Chargement des données...
  Train: (13536, 17) | Test: (5621, 14) | Prior: (44882, 17)
  07_days: 1.130%
  90_days: 1.581%
  20_days: 2.231%

[2] Stats Prior...
  Calcul des stats Prior...

[3] Feature Engineering...
  Features: 294

[4] Entraînement (3 targets, n_jobs=3)...

══════ adopted_within_90_days ══════

══════ adopted_within_07_days ══════

══════ adopted_within_120_days ══════
  Features sélectionnées: 150/294
    Fold 4/10 START
    Fold 7/10 START
    Fold 2/10 START
    Fold 3/10 START
    Fold 6/10 START
    Fold 8/10 START
    Fold 1/10 START
    Fold 5/10 START
  Features sélectionnées: 150/294
    Fold 7/10 START
    Fold 3/10 START
    Fold 8/10 START
    Fold 5/10 START
    Fold 2/10 START
    Fold 1/10 START
    Fold 4/10 START
    Fold 6/10 START
  Features sélec